In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

pd.set_option("display.max_columns", None)

print("Libraries imported successfully!")

In [ ]:
df = pd.read_csv("../data/processed/merged_f1_data.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
print(df.dtypes)
df.info()

In [ ]:
df.describe().T

In [ ]:
missing_values = df.isnull().sum()

missing_values = missing_values[
    missing_values > 0
].sort_values(ascending=False)

print("Columns containing missing values:")
display(missing_values)

print("\nTotal missing values:",
      df.isnull().sum().sum())

In [ ]:
print(
    "Missing qualifying positions before:",
    df["qualifying_position"].isnull().sum()
)

df["qualifying_position"] = (
    df["qualifying_position"].fillna(df["grid"])
)

print(
    "Missing qualifying positions after:",
    df["qualifying_position"].isnull().sum()
)

In [ ]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

race_driver_duplicates = df.duplicated(
    subset=["season", "round", "driver_id"]
).sum()

print(
    "Duplicate season-round-driver records:",
    race_driver_duplicates
)

In [ ]:
numeric_columns = [
    "season",
    "round",
    "number",
    "grid",
    "position",
    "points",
    "laps",
    "fastest_lap_rank",
    "fastest_lap_speed",
    "qualifying_position"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

print("Data types cleaned.")

In [ ]:
print("Invalid season values:")
display(
    df.loc[
        ~df["season"].between(2010, 2025),
        ["season"]
    ].drop_duplicates()
)

print("\nInvalid grid values:")
display(
    df.loc[
        df["grid"] < 0,
        ["grid"]
    ].drop_duplicates()
)

print("\nInvalid points values:")
display(
    df.loc[
        df["points"] < 0,
        ["points"]
    ].drop_duplicates()
)

print("\nInvalid laps values:")
display(
    df.loc[
        df["laps"] < 0,
        ["laps"]
    ].drop_duplicates()
)

In [ ]:
df = df.sort_values(
    ["season", "round", "date", "driver_id"]
).reset_index(drop=True)

print("Dataset sorted chronologically.")

display(
    df[
        [
            "season",
            "round",
            "race_name",
            "driver_name"
        ]
    ].head(10)
)

In [ ]:
df["previous_finish"] = (
    df.groupby("driver_id")["position"]
      .shift(1)
)

df[
    [
        "season",
        "round",
        "driver_name",
        "position",
        "previous_finish"
    ]
].head(20)

In [ ]:
df["avg_finish_all_previous"] = (
    df.groupby("driver_id")["position"]
      .transform(
          lambda x:
          x.shift(1).expanding().mean()
      )
)

df["avg_qualifying_all_previous"] = (
    df.groupby("driver_id")["qualifying_position"]
      .transform(
          lambda x:
          x.shift(1).expanding().mean()
      )
)


In [ ]:
df["avg_finish_last_5"] = (
    df.groupby("driver_id")["position"]
      .transform(
          lambda x:
          x.shift(1)
           .rolling(5, min_periods=1)
           .mean()
      )
)

df["avg_qualifying_last_5"] = (
    df.groupby("driver_id")["qualifying_position"]
      .transform(
          lambda x:
          x.shift(1)
           .rolling(5, min_periods=1)
           .mean()
      )
)

In [ ]:
df["season_points_so_far"] = (
    df.groupby(
        ["season", "driver_id"]
    )["points"]
    .transform(
        lambda x:
        x.shift(1)
         .fillna(0)
         .cumsum()
    )
)

In [ ]:
df["season_wins_so_far"] = (
    df.groupby(
        ["season", "driver_id"]
    )["position"]
    .transform(
        lambda x:
        (x.shift(1) == 1)
        .fillna(False)
        .cumsum()
    )
)

In [ ]:
df["season_podiums_so_far"] = (
    df.groupby(
        ["season", "driver_id"]
    )["position"]
    .transform(
        lambda x:
        (x.shift(1) <= 3)
        .fillna(False)
        .cumsum()
    )
)

In [ ]:
df["constructor_points_so_far"] = (
    df.groupby(
        ["season", "constructor"]
    )["points"]
    .transform(
        lambda x:
        x.shift(1)
         .fillna(0)
         .cumsum()
    )
)

In [ ]:
df["constructor_wins_so_far"] = (
    df.groupby(
        ["season", "constructor"]
    )["position"]
    .transform(
        lambda x:
        (x.shift(1) == 1)
        .fillna(False)
        .cumsum()
    )
)

In [ ]:
df["constructor_podiums_so_far"] = (
    df.groupby(
        ["season", "constructor"]
    )["position"]
    .transform(
        lambda x:
        (x.shift(1) <= 3)
        .fillna(False)
        .cumsum()
    )
)

In [ ]:
df["career_races_before"] = (
    df.groupby("driver_id").cumcount()
)

print(
    df[
        [
            "driver_name",
            "season",
            "round",
            "career_races_before"
        ]
    ].head(20)
)

In [ ]:
historical_columns = [
    "previous_finish",
    "avg_finish_all_previous",
    "avg_qualifying_all_previous",
    "avg_finish_last_5",
    "avg_qualifying_last_5"
]

df["previous_finish"] = (
    df["previous_finish"]
    .fillna(df["avg_finish_all_previous"])
)

df["avg_finish_last_5"] = (
    df["avg_finish_last_5"]
    .fillna(df["avg_finish_all_previous"])
)

df["avg_qualifying_last_5"] = (
    df["avg_qualifying_last_5"]
    .fillna(df["avg_qualifying_all_previous"])
)

for column in historical_columns:
    df[column] = df[column].fillna(20)

print(
    df[historical_columns].isnull().sum()
)

In [ ]:
df["podium"] = (
    df["position"] <= 3
).astype(int)

print("Podium distribution:")
print(df["podium"].value_counts())

print("\nPodium percentage:")
print(
    df["podium"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
feature_columns = [
    "qualifying_position",
    "grid",
    "previous_finish",
    "avg_finish_all_previous",
    "avg_qualifying_all_previous",
    "avg_finish_last_5",
    "avg_qualifying_last_5",
    "season_points_so_far",
    "season_wins_so_far",
    "season_podiums_so_far",
    "constructor_points_so_far",
    "constructor_wins_so_far",
    "constructor_podiums_so_far",
    "career_races_before"
]

X = df[feature_columns].copy()

y_regression = df["position"].copy()

y_classification = df["podium"].copy()

print("X shape:", X.shape)
print("Regression target:", y_regression.shape)
print("Classification target:", y_classification.shape)

In [ ]:
outlier_report = []

for column in feature_columns:

    Q1 = X[column].quantile(0.25)
    Q3 = X[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = (
        (X[column] < lower_bound) |
        (X[column] > upper_bound)
    ).sum()

    outlier_report.append({
        "Feature": column,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": outliers
    })

outlier_report = pd.DataFrame(outlier_report)

display(
    outlier_report.sort_values(
        "Outlier Count",
        ascending=False
    )
)

In [ ]:
train_mask = df["season"].between(2010, 2023)

validation_mask = (
    df["season"] == 2024
)

test_mask = (
    df["season"] == 2025
)

In [ ]:
X_train = df.loc[
    train_mask,
    feature_columns
].copy()

X_validation = df.loc[
    validation_mask,
    feature_columns
].copy()

X_test = df.loc[
    test_mask,
    feature_columns
].copy()


y_reg_train = df.loc[
    train_mask,
    "position"
].copy()

y_reg_validation = df.loc[
    validation_mask,
    "position"
].copy()

y_reg_test = df.loc[
    test_mask,
    "position"
].copy()


y_clf_train = df.loc[
    train_mask,
    "podium"
].copy()

y_clf_validation = df.loc[
    validation_mask,
    "podium"
].copy()

y_clf_test = df.loc[
    test_mask,
    "podium"
].copy()

In [ ]:
print("TRAINING")
print("Seasons: 2010–2023")
print("Rows:", len(X_train))

print("\nVALIDATION")
print("Season: 2024")
print("Rows:", len(X_validation))

print("\nTEST")
print("Season: 2025")
print("Rows:", len(X_test))

In [ ]:
print("Training missing values:")
print(X_train.isnull().sum().sum())

print("\nValidation missing values:")
print(X_validation.isnull().sum().sum())

print("\nTest missing values:")
print(X_test.isnull().sum().sum())

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_validation_scaled = scaler.transform(
    X_validation
)

X_test_scaled = scaler.transform(
    X_test
)

print("Scaling completed.")
print("Scaler fitted only on training data.")

In [ ]:
scaled_train = pd.DataFrame(
    X_train_scaled,
    columns=feature_columns
)

print("Training means:")
display(
    scaled_train.mean().round(4)
)

print("\nTraining standard deviations:")
display(
    scaled_train.std().round(4)
)

In [ ]:
print("=" * 60)
print("FINAL PREPROCESSING SUMMARY")
print("=" * 60)

print("\nOriginal dataset:", df.shape)

print("\nFeatures:", len(feature_columns))

print("\nTraining:", X_train.shape)
print("Validation:", X_validation.shape)
print("Testing:", X_test.shape)

print("\nRegression target:")
print(y_reg_train.shape)

print("\nClassification target:")
print(y_clf_train.shape)

print("\nMissing values:")
print(
    X_train.isnull().sum().sum(),
    X_validation.isnull().sum().sum(),
    X_test.isnull().sum().sum()
)

In [34]:
OUTPUT_DIR = "../data/processed"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# Full processed master dataset
df.to_csv(
    f"{OUTPUT_DIR}/f1_master_dataset_final.csv",
    index=False
)

# Training / validation / testing
df.loc[train_mask].to_csv(
    f"{OUTPUT_DIR}/train_final.csv",
    index=False
)

df.loc[validation_mask].to_csv(
    f"{OUTPUT_DIR}/validation_final.csv",
    index=False
)

df.loc[test_mask].to_csv(
    f"{OUTPUT_DIR}/test_final.csv",
    index=False
)

# Scaled feature matrices
pd.DataFrame(
    X_train_scaled,
    columns=feature_columns
).to_csv(
    f"{OUTPUT_DIR}/X_train_scaled.csv",
    index=False
)

pd.DataFrame(
    X_validation_scaled,
    columns=feature_columns
).to_csv(
    f"{OUTPUT_DIR}/X_validation_scaled.csv",
    index=False
)

pd.DataFrame(
    X_test_scaled,
    columns=feature_columns
).to_csv(
    f"{OUTPUT_DIR}/X_test_scaled.csv",
    index=False
)

print("All datasets saved successfully!")

All datasets saved successfully!
